# Fine-tuning do Qwen3-4B para a Indie com Unsloth (QLoRA)

**Projeto final:** Generative AI & Advanced Analytics

---

Este notebook treina o assistente virtual da **Indie**, fintech fictícia para quem
trabalha por conta própria, a partir da base de perguntas e respostas gerada no
notebook `02_geracao_base_qna.ipynb`. A técnica é **PEFT com QLoRA**: o Qwen3-4B é
carregado em 4 bits e congelado, e apenas adaptadores LoRA de baixo rank são
treinados. Isso cabe numa GPU T4 do Colab.

O pipeline segue o notebook de aula `fine_tuning_qwen3_aula.ipynb`, com as mesmas
funções auxiliares e a mesma sequência de etapas. As principais adaptações foram:

| Item | Aula (SynapseAI) | Este projeto (Indie) | Motivo |
|---|---|---|---|
| Dataset | `synapseai_knowledge_base.jsonl` | `indie_knowledge_base.jsonl` | Base própria da empresa |
| Prompt de sistema | Definido no código | Lido do próprio dataset | Garante que treino e inferência usem exatamente o mesmo prompt |
| `MAX_SEQ_LENGTH` | 2048 | 1024 | Os exemplos têm menos de 700 tokens (verificado na seção 1.6); menos memória |
| Épocas | 1 | 3 | Base pequena (~500 exemplos de treino): uma época dá só ~60 passos, pouco para fixar preços e prazos |
| Melhor checkpoint | — | `load_best_model_at_end` pela perda de validação | Protege contra overfitting nas épocas finais |
| Avaliação qualitativa | 2 perguntas após o treino | 10 perguntas **antes e depois** do treino | Mostra o que o modelo aprendeu sobre o domínio |
| Publicação | `push_to_hub` do modelo e do dataset formatado | Adaptadores e model card; dataset `.jsonl` e dataset card em um repositório separado | Atende ao item 2.4 do enunciado |

**Organização:** a Parte 1 cobre o treinamento e a publicação. A Parte 2 cobre a
inferência a partir do modelo publicado no Hub.

**Antes de executar:**
- Use um ambiente com GPU (*Ambiente de execução → Alterar o tipo → T4 GPU*).
- Cadastre nos *secrets* do Colab um `HF_TOKEN` com permissão de **escrita**.
- Confira `HF_USERNAME` na classe `Config` (`EstudoAI`).

## Parte 1 - Treinamento do modelo

### 1.1 Preparação do ambiente

A célula abaixo instala o Unsloth, que já traz `transformers`, `trl`, `peft` e
`bitsandbytes`, usando o `uv`, como no notebook de aula.

In [ ]:
!uv pip install unsloth psutil -qqq

O Unsloth precisa ser importado **antes** de `transformers` e `trl` para aplicar as
otimizações dele. Por isso, fica em uma célula própria.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

from unsloth import FastLanguageModel

print("Unsloth carregado com sucesso.")

### 1.2 Configuração do treinamento

A classe `Config` centraliza todos os hiperparâmetros e caminhos. A tabela abaixo
justifica as escolhas.

**Modelo e dados**

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `MODEL_NAME` | `unsloth/Qwen3-4B` | Mesmo modelo da aula: bom em português, suporta *chat template* com papéis e cabe numa T4 em 4 bits |
| `LOAD_IN_4BIT` | `True` | QLoRA: o modelo base fica quantizado em 4 bits e congelado, usando cerca de 3,5 GB de VRAM |
| `MAX_SEQ_LENGTH` | 1024 | O maior exemplo tem menos de 700 tokens |
| `EVAL_SIZE` | 0.10 | ~55 exemplos para a curva de validação, conforme o enunciado |
| `ENABLE_THINKING` | `False` | O assistente responde direto, sem o modo de raciocínio do Qwen3 |

**LoRA**

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `LORA_RANK` (r) | 16 | Capacidade suficiente para um domínio restrito (~33 M parâmetros treináveis, menos de 1% do modelo). Ranks maiores aumentam o risco de overfitting numa base pequena |
| `LORA_ALPHA` | 16 | Com rsLoRA, a escala efetiva é α/√r = 4, e as atualizações continuam estáveis mesmo com rank maior |
| `USE_RSLORA` | `True` | *Rank-stabilized LoRA*, como na aula |
| `LORA_DROPOUT` | 0 | Caminho otimizado do Unsloth. O overfitting é controlado pela validação e pelo melhor checkpoint |
| `TARGET_MODULES` | atenção (q, k, v, o) + MLP (gate, up, down) | Todas as camadas lineares. As MLPs são onde fica boa parte do conhecimento factual (preços, prazos, regras) |

**Treinamento**

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `BATCH_SIZE` × `GRAD_ACCUM` | 2 × 4 = **8** | Batch efetivo 8 com pouca memória por passo |
| `EPOCHS` | 3 | ~500 exemplos / 8 = ~63 passos por época, ou ~190 no total |
| `LEARNING_RATE` | 1e-4 | Valor da aula. Com rsLoRA, a escala do adaptador já é 4×, então não é preciso subir para 2e-4 |
| `LR_SCHEDULER` / `WARMUP_RATIO` | cosine / 0.05 | Aquecimento curto e decaimento suave até o fim da 3ª época |
| `WEIGHT_DECAY` / `MAX_GRAD_NORM` | 0.01 / 1.0 | Regularização leve e *gradient clipping* |
| `EVAL_STEPS` | 20 | ~9 pontos na curva de validação |
| `LOAD_BEST_MODEL_AT_END` | `True` | Restaura o checkpoint com a menor perda de validação |

In [ ]:
import json
import math
import os
from pathlib import Path
from string import Template
from typing import Any, Optional

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi
from transformers import TextStreamer
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only


class Config:
    """Central configuration for the fine-tuning run.

    This class groups model, dataset, LoRA and training
    hyperparameters in a single place, making the experiment
    easier to reproduce and adjust.
    """

    MODEL_NAME = "unsloth/Qwen3-4B"
    MAX_SEQ_LENGTH = 1024
    LOAD_IN_4BIT = True

    USE_GOOGLE_DRIVE = True
    DRIVE_DIR = "/content/drive/MyDrive/PUC_GenAI_Indie"
    LOCAL_DIR = "indie_ft"
    DATA_FILE = "indie_knowledge_base.jsonl"
    METADATA_FILE = "indie_qna_com_metadados.jsonl"
    PIPELINE_SUMMARY_FILE = "resumo_pipeline.json"
    USE_EVAL_SPLIT = True
    EVAL_SIZE = 0.10
    ENABLE_THINKING = False

    LORA_RANK = 16
    LORA_ALPHA = 16
    LORA_DROPOUT = 0
    USE_RSLORA = True
    TARGET_MODULES = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]

    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    EPOCHS = 3
    LEARNING_RATE = 1e-4
    WARMUP_RATIO = 0.05
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0
    LR_SCHEDULER = "cosine"
    LOGGING_STEPS = 5
    EVAL_STEPS = 20
    LOAD_BEST_MODEL_AT_END = True
    SEED = 3407
    OUTPUT_DIR = "outputs"

    MAX_NEW_TOKENS = 512
    TEMPERATURE = 0.1

    PUSH_TO_HUB = True
    HF_USERNAME = "EstudoAI"
    HF_PRIVATE = False
    MODEL_REPO = "Indie-Qwen3-4B-LoRA"
    DATASET_REPO = "Indie-Knowledge-Base"

    def __init__(self) -> None:
        self.base_dir = Path(
            self.DRIVE_DIR if self.USE_GOOGLE_DRIVE else self.LOCAL_DIR
        )

    def path(self, filename: str) -> Path:
        """Return the full path of a project file inside the base dir."""
        return self.base_dir / filename

    @property
    def model_repo_id(self) -> str:
        """Hugging Face Hub id of the model repository."""
        return f"{self.HF_USERNAME}/{self.MODEL_REPO}"

    @property
    def dataset_repo_id(self) -> str:
        """Hugging Face Hub id of the dataset repository."""
        return f"{self.HF_USERNAME}/{self.DATASET_REPO}"

A célula abaixo monta o Google Drive, onde estão a base gerada no notebook 02 e onde
serão salvos os artefatos (curva de perda, exemplos, adaptadores). Também mostra qual
GPU está em uso.

In [ ]:
config = Config()

if config.USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
config.base_dir.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Ative uma GPU no Colab antes de continuar."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Artefatos em: {config.base_dir}")

### 1.3 Carregamento do dataset

A base é lida do Drive. Se não estiver lá, o Colab abre o upload. Cada linha tem uma
lista `messages` com os papéis `system`, `user` e `assistant`. O prompt de sistema
padrão, usado nas inferências, é lido do próprio dataset.

In [ ]:
data_path = config.path(config.DATA_FILE)
if not data_path.exists():
    from google.colab import files

    print(f"Envie o arquivo {config.DATA_FILE}:")
    uploaded = files.upload()
    data_path.write_bytes(next(iter(uploaded.values())))

raw_dataset = load_dataset("json", data_files=str(data_path), split="train")
DEFAULT_SYSTEM_PROMPT = raw_dataset[0]["messages"][0]["content"]

print(f"{len(raw_dataset)} exemplos carregados.")
print("\nPrompt de sistema:\n")
print(DEFAULT_SYSTEM_PROMPT)
print("\nExemplo:")
for message in raw_dataset[1]["messages"][1:]:
    print(f"[{message['role']}] {message['content']}")

### 1.4 Funções auxiliares

As funções abaixo são as mesmas do notebook de aula, com pequenas adaptações:

- `format_dataset`: aplica o *chat template* do Qwen3 a cada exemplo;
- `plot_loss`: plota as curvas de perda de treino e de validação e salva a figura no
  Drive;
- `generate_response`: versão de `predict_response` que também **retorna** o texto
  gerado, para montar a comparação antes/depois;
- `get_hf_token`: lê o token do Hugging Face dos *secrets* do Colab.

In [ ]:
def format_dataset(
    example: dict,
    tokenizer: Any,
    config: Config,
) -> dict:
    """Apply the Qwen3 chat template to a single dataset example.

    Args:
        example: A single dataset row containing a 'messages' key
            with the chat-formatted conversation (list of dicts
            with 'role' and 'content' keys).
        tokenizer: The tokenizer associated with the base model,
            used to render the chat template.
        config: The training configuration, used to read whether
            the Qwen3 'thinking' mode should be enabled.

    Returns:
        A dictionary with a single 'text' key containing the
        rendered chat string.
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=config.ENABLE_THINKING,
    )
    return {"text": text}


def plot_loss(trainer: Any, output_path: Path) -> None:
    """Plot the training and evaluation loss curves.

    Reads the trainer's log history, extracts the training and
    (when available) evaluation loss values per step, plots both
    curves and saves the resulting figure.

    Args:
        trainer: A trained Hugging Face / TRL trainer instance
            exposing a 'state.log_history' attribute.
        output_path: Where to save the figure (PNG).

    Returns:
        None. The plot is displayed and saved to disk as a side
        effect.
    """
    log_history = trainer.state.log_history

    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []

    for log in log_history:
        if "loss" in log and "step" in log:
            train_steps.append(log["step"])
            train_losses.append(log["loss"])
        if "eval_loss" in log and "step" in log:
            eval_steps.append(log["step"])
            eval_losses.append(log["eval_loss"])

    plt.figure(figsize=(10, 6))
    plt.plot(
        train_steps,
        train_losses,
        label="Training Loss",
        color="blue",
        marker="o",
        markersize=4,
    )
    if eval_losses:
        plt.plot(
            eval_steps,
            eval_losses,
            label="Eval Loss",
            color="red",
            marker="s",
            markersize=4,
        )
    plt.xlabel("Steps (Passos de Treinamento)")
    plt.ylabel("Loss (Perda)")
    plt.title("Curva de Convergencia - Indie Qwen3-4B QLoRA")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig(output_path, dpi=120)
    plt.show()
    print(f"Grafico salvo em '{output_path}'")


def generate_response(
    model: Any,
    tokenizer: Any,
    config: Config,
    question: str,
    system_prompt: Optional[str] = None,
    stream: bool = False,
) -> str:
    """Generate a model response for a given question.

    Builds the chat prompt using the system and user messages and
    generates an answer, optionally streaming the tokens to standard
    output. The model must already be in inference mode.

    Args:
        model: The language model (base or fine-tuned).
        tokenizer: The tokenizer associated with the model.
        config: The training configuration (thinking mode,
            generation length and temperature).
        question: The user question to send to the model.
        system_prompt: An optional system prompt overriding the
            default persona. Defaults to 'DEFAULT_SYSTEM_PROMPT'.
        stream: Whether to stream the answer to standard output.

    Returns:
        The generated answer as a string.
    """
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=config.ENABLE_THINKING,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None

    output = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=config.MAX_NEW_TOKENS,
        temperature=config.TEMPERATURE,
        use_cache=True,
    )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def answer_questions(
    model: Any,
    tokenizer: Any,
    config: Config,
    questions: list[str],
) -> list[str]:
    """Answer a list of questions in inference mode and restore training mode.

    Args:
        model: The language model with LoRA adapters.
        tokenizer: The tokenizer associated with the model.
        config: The training configuration.
        questions: The questions to answer.

    Returns:
        The generated answers, in the same order as the questions.
    """
    FastLanguageModel.for_inference(model)
    answers = []
    for question in questions:
        answer = generate_response(model, tokenizer, config, question)
        print(f"Usuario: {question}\nAssistente: {answer}\n" + "-" * 80)
        answers.append(answer)
    FastLanguageModel.for_training(model)
    return answers


def get_hf_token() -> str:
    """Fetch the Hugging Face authentication token.

    Attempts to read the token from Google Colab secrets first,
    falling back to the 'HF_TOKEN' environment variable when the
    Colab API is not available.

    Returns:
        The Hugging Face authentication token as a string.

    Raises:
        ValueError: If the token cannot be found in either the
            Colab secrets or the environment variables.
    """
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
        if not token:
            raise ValueError(
                "Token nao encontrado. Configure o HF_TOKEN nos "
                "segredos do Colab ou exporte a variavel de "
                "ambiente HF_TOKEN."
            )
        return token

### 1.5 Carregamento do modelo base e aplicação do LoRA

O Qwen3-4B é carregado em 4 bits (QLoRA) e, em seguida, recebe os adaptadores LoRA.
Só os adaptadores são treináveis. A saída mostra quantos parâmetros são treinados em
relação ao total.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.MODEL_NAME,
    max_seq_length=config.MAX_SEQ_LENGTH,
    load_in_4bit=config.LOAD_IN_4BIT,
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=config.LORA_RANK,
    target_modules=config.TARGET_MODULES,
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    bias="none",
    use_rslora=config.USE_RSLORA,
    use_gradient_checkpointing="unsloth",
    random_state=config.SEED,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(
    f"Parametros treinaveis: {trainable_params:,} de {total_params:,} "
    f"({100 * trainable_params / total_params:.2f}%)"
)

### 1.6 Formatação do dataset e divisão treino/validação

Cada conversa é convertida em texto com o *chat template* do Qwen3. A célula também
mede o tamanho dos exemplos em tokens, para confirmar que nenhum passa de
`MAX_SEQ_LENGTH`, e separa 10% da base para validação.

In [ ]:
dataset = raw_dataset.map(lambda example: format_dataset(example, tokenizer, config))

token_lengths = pd.Series([len(tokenizer(text)["input_ids"]) for text in dataset["text"]])
print("Tamanho dos exemplos (tokens):")
print(token_lengths.describe(percentiles=[0.5, 0.95]).round(0).to_string())
assert token_lengths.max() <= config.MAX_SEQ_LENGTH, "Aumente MAX_SEQ_LENGTH."

eval_dataset = None
if config.USE_EVAL_SPLIT:
    split = dataset.train_test_split(test_size=config.EVAL_SIZE, seed=config.SEED)
    train_dataset, eval_dataset = split["train"], split["test"]
    print(f"\nTreino: {len(train_dataset)} | Validacao: {len(eval_dataset)}")
else:
    train_dataset = dataset

steps_per_epoch = math.ceil(len(train_dataset) / (config.BATCH_SIZE * config.GRAD_ACCUM))
print(f"Passos por epoca: {steps_per_epoch} | Total: {steps_per_epoch * config.EPOCHS}")

print("\n----- Exemplo formatado -----")
print(train_dataset[0]["text"])

### 1.7 Respostas do modelo base (antes do treino)

Logo após `get_peft_model`, a matriz B dos adaptadores LoRA é inicializada com zeros.
Por isso, o modelo ainda se comporta **exatamente como o Qwen3-4B original**. As
perguntas abaixo são respondidas agora e de novo depois do treino. Elas cobrem
produtos, planos, taxas, regulação, golpes, limites da assistente e perguntas fora do
escopo, e várias foram escritas de um jeito diferente das perguntas da base.

In [ ]:
TEST_QUESTIONS = [
    "Quais produtos a Indie oferece?",
    "Qual a diferença entre o plano Pro e o Premium?",
    "Sou tradutora e recebo em euro de clientes na Alemanha. Quanto a Indie cobra para converter?",
    "O dinheiro que fica parado na conta da Indie tem garantia do FGC?",
    "Me ligaram dizendo ser da Indie e pediram o código da Indie Chave para cancelar uma compra. O que eu faço?",
    "vcs tem maquininha de cartao?",
    "Em qual ação eu deveria investir o dinheiro da minha empresa?",
    "Como faço para abrir um MEI pelo app?",
    "Consigo pagar um fornecedor nos Estados Unidos pela Indie?",
    "Qual é a capital da Austrália?",
]

answers_before = answer_questions(model, tokenizer, config, TEST_QUESTIONS)

### 1.8 Treinamento supervisionado (SFT)

O `SFTTrainer` é configurado como na aula, com duas diferenças: salva checkpoints no
mesmo ritmo da avaliação e restaura o melhor pela perda de validação
(`load_best_model_at_end`).

`train_on_responses_only` mascara o prompt de sistema e a pergunta do usuário no
cálculo da perda. Assim, o modelo aprende só a **gerar as respostas** da assistente,
e não a reproduzir o prompt de sistema, que se repete em todos os exemplos.

In [ ]:
sft_args = dict(
    dataset_text_field="text",
    per_device_train_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRAD_ACCUM,
    warmup_ratio=config.WARMUP_RATIO,
    num_train_epochs=config.EPOCHS,
    learning_rate=config.LEARNING_RATE,
    logging_steps=config.LOGGING_STEPS,
    optim="adamw_8bit",
    weight_decay=config.WEIGHT_DECAY,
    max_grad_norm=config.MAX_GRAD_NORM,
    lr_scheduler_type=config.LR_SCHEDULER,
    seed=config.SEED,
    output_dir=config.OUTPUT_DIR,
    report_to="none",
)
if eval_dataset is not None:
    sft_args.update(
        eval_strategy="steps",
        eval_steps=config.EVAL_STEPS,
        per_device_eval_batch_size=config.BATCH_SIZE,
    )
    if config.LOAD_BEST_MODEL_AT_END:
        sft_args.update(
            save_strategy="steps",
            save_steps=config.EVAL_STEPS,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        )

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(**sft_args),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Antes de treinar, a célula abaixo decodifica um exemplo **com a máscara aplicada**.
Os tokens ignorados na perda aparecem como espaços, e só a resposta da assistente
deve continuar visível. Essa é a verificação recomendada pelo Unsloth para
`train_on_responses_only`.

In [ ]:
sample_labels = trainer.train_dataset[0]["labels"]
visible = tokenizer.decode(
    [token if token != -100 else tokenizer.pad_token_id for token in sample_labels],
    skip_special_tokens=True,
)
print(visible.strip())

In [ ]:
train_result = trainer.train()
print(f"\nTreinamento concluido em {train_result.metrics['train_runtime'] / 60:.1f} min.")

### 1.9 Curva de perda e avaliação

Uma curva saudável mostra as duas perdas caindo. Se a perda de validação voltar a
subir enquanto a de treino continua caindo, há overfitting. Nesse caso,
`load_best_model_at_end` já restaurou o checkpoint com a menor perda de validação.
Essa perda (`trainer.state.best_metric`) é a que aparece no resumo.

In [ ]:
plot_loss(trainer, config.path("training_loss.png"))

# Nao chamamos trainer.evaluate() aqui: nas versoes recentes do transformers, a
# barra de progresso do notebook e descartada ao fim do train() e o evaluate()
# numa celula separada falha ("on_train_begin must be called before
# on_evaluate"). Como metric_for_best_model="eval_loss", o best_metric ja e a
# perda de validacao do checkpoint restaurado.
train_losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in trainer.state.log_history if "eval_loss" in log]
best_eval_loss = trainer.state.best_metric or (min(eval_losses) if eval_losses else float("nan"))

training_summary = {
    "modelo_base": config.MODEL_NAME,
    "gpu": torch.cuda.get_device_name(0),
    "exemplos_treino": len(train_dataset),
    "exemplos_validacao": len(eval_dataset) if eval_dataset is not None else 0,
    "parametros_treinaveis": trainable_params,
    "percentual_treinavel": round(100 * trainable_params / total_params, 3),
    "lora_rank": config.LORA_RANK,
    "lora_alpha": config.LORA_ALPHA,
    "lora_dropout": config.LORA_DROPOUT,
    "use_rslora": config.USE_RSLORA,
    "target_modules": config.TARGET_MODULES,
    "batch_size_efetivo": config.BATCH_SIZE * config.GRAD_ACCUM,
    "epocas": config.EPOCHS,
    "learning_rate": config.LEARNING_RATE,
    "scheduler": config.LR_SCHEDULER,
    "passos_totais": trainer.state.global_step,
    "tempo_treino_min": round(train_result.metrics["train_runtime"] / 60, 1),
    "loss_treino_inicial": round(train_losses[0], 4),
    "loss_treino_final": round(train_losses[-1], 4),
    "eval_loss_inicial": round(eval_losses[0], 4) if eval_losses else None,
    "melhor_eval_loss": round(best_eval_loss, 4),
    "melhor_checkpoint": trainer.state.best_model_checkpoint,
}
config.path("treino_resumo.json").write_text(
    json.dumps(training_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))

### 1.10 Respostas do modelo treinado (depois do treino)

As mesmas perguntas da seção 1.7 são respondidas pelo modelo treinado, e a tabela
compara as duas versões. O arquivo `exemplos_inferencia.csv` é salvo no Drive para o
relatório.

In [ ]:
answers_after = answer_questions(model, tokenizer, config, TEST_QUESTIONS)

comparison = pd.DataFrame(
    {
        "pergunta": TEST_QUESTIONS,
        "antes_do_treino": answers_before,
        "depois_do_treino": answers_after,
    }
)
comparison.to_csv(config.path("exemplos_inferencia.csv"), index=False, encoding="utf-8-sig")

with pd.option_context("display.max_colwidth", None):
    display(comparison.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

### 1.11 Salvando os adaptadores no Drive

Uma cópia dos adaptadores LoRA e do tokenizer fica no Drive, para que o modelo não se
perca se o upload para o Hub falhar ou o Colab desconectar.

In [ ]:
adapters_dir = config.path("indie_lora_adapters")
model.save_pretrained(adapters_dir)
tokenizer.save_pretrained(adapters_dir)
print(f"Adaptadores salvos em: {adapters_dir}")
print(sorted(p.name for p in adapters_dir.iterdir()))

### 1.12 Publicação no Hugging Face Hub

As funções abaixo publicam dois repositórios **públicos** e separados:

1. **Modelo** (`Indie-Qwen3-4B-LoRA`): adaptadores LoRA e tokenizer, com um model card
   (`README.md`) que descreve a empresa, o modelo base, os hiperparâmetros, os
   resultados e como carregar o modelo para inferência.
2. **Dataset** (`Indie-Knowledge-Base`): o arquivo `.jsonl` usado no treinamento, no
   formato `messages`, com um dataset card que explica como a base foi construída. A
   versão com metadados (tema e estilo de cada par) também é enviada, se existir.

In [ ]:
MODEL_CARD_TEMPLATE = Template("""---
base_model: $base_model
library_name: peft
language:
- pt
license: apache-2.0
pipeline_tag: text-generation
datasets:
- $dataset_repo_id
tags:
- lora
- qlora
- unsloth
- trl
- sft
- fintech
---

# Indie-Qwen3-4B-LoRA

Adaptadores **LoRA** treinados sobre o **$model_name** para atuar como assistente virtual da
**Indie**, uma fintech **fictícia** criada para o trabalho final da disciplina
*Generative AI & Advanced Analytics*.

## Caso de uso: a Indie

A Indie é "o banco de quem trabalha por conta própria": uma fintech que oferece serviços financeiros
para MEIs, microempresas de serviços, freelancers e profissionais liberais. O modelo responde sobre:

- **Indie Conta PJ**: conta digital PJ com Pix, cartões e abertura de MEI (Indie Abre MEI);
- **Indie Notas**: emissão de nota fiscal de serviço (NFS-e);
- **Indie Global**: recebimento do exterior em USD, EUR e GBP;
- **Cofre Fiscal**: reserva automática para impostos e DAS-MEI automático;
- **Indie Contábil**: contabilidade online com contador dedicado;
- **Indie Futuro**: Caixinhas, Salário Indie, Seguro Renda Protegida e Previdência;
- **Indie Cobranças**: links de pagamento, cobrança recorrente e Tap to Pay;
- planos (Essencial, Pro e Premium), atendimento, segurança e os limites da própria assistente.

## Treinamento

- **Modelo base:** `$model_name` (carregado em 4 bits: `$base_model`)
- **Técnica:** SFT com QLoRA (PEFT) via [Unsloth](https://github.com/unslothai/unsloth) e TRL, com perda calculada só nas respostas da assistente (`train_on_responses_only`)
- **Dataset:** [$dataset_repo_id](https://huggingface.co/datasets/$dataset_repo_id), com $n_train exemplos de treino e $n_eval de validação

| Hiperparâmetro | Valor |
|---|---|
| LoRA rank (r) / alpha / dropout | $lora_rank / $lora_alpha / $lora_dropout |
| rsLoRA | $use_rslora |
| Módulos-alvo | $target_modules |
| Parâmetros treináveis | $trainable_params ($trainable_pct% do total) |
| Batch efetivo | $batch_size (batch $per_device × acumulação $grad_accum) |
| Épocas / passos | $epochs / $steps |
| Learning rate / scheduler | $learning_rate / $scheduler (warmup $warmup) |
| Comprimento máximo | $max_seq_length tokens |
| GPU / tempo de treino | $gpu / $runtime min |

**Perdas:** treino $loss_start → $loss_end; melhor perda de validação: **$eval_loss**.

## Como usar

### Com Unsloth (recomendado)

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="$model_repo_id",
    max_seq_length=$max_seq_length,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},  # prompt de sistema abaixo
    {"role": "user", "content": "Quais produtos a Indie oferece?"},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

### Com Transformers + PEFT

```python
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained("$model_repo_id", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("$model_repo_id")
```

Requer GPU com `bitsandbytes`, porque o modelo base de referência está quantizado em 4 bits.

### Prompt de sistema usado no treinamento

Use o mesmo prompt de sistema do treinamento para obter o comportamento esperado:

```text
$system_prompt
```

## Exemplos (modelo treinado)

$examples

## Limitações

- A Indie é uma **empresa fictícia**. Produtos, preços e parceiros não existem, e o modelo **não** deve ser usado
  para orientação financeira ou tributária real.
- O modelo pode errar valores ou combinar informações de forma incorreta, sobretudo em perguntas muito distantes da base de treino.
""")

DATASET_CARD_TEMPLATE = Template("""---
language:
- pt
license: apache-2.0
task_categories:
- text-generation
- question-answering
size_categories:
- n<1K
pretty_name: Indie Knowledge Base
tags:
- fintech
- synthetic
- instruction-tuning
- chat
configs:
- config_name: default
  data_files: $data_file
---

# Indie Knowledge Base

Base de **$n_examples conversas** (pergunta e resposta) sobre a **Indie**, fintech **fictícia** para
quem trabalha por conta própria. Foi usada para treinar o modelo
[$model_repo_id](https://huggingface.co/$model_repo_id) no trabalho final da disciplina
*Generative AI & Advanced Analytics*.

## Formato

Arquivo `$data_file`: uma conversa por linha, no formato `messages` com os papéis `system`, `user` e `assistant`.

```json
$example
```

$metadata_note

## Como a base foi construída

A base foi **gerada com apoio de LLM e revisada**, a partir de um documento de referência da empresa
(produtos, planos, taxas, suporte, segurança e diretrizes da assistente):

1. **Geração** com `gpt-4o-mini` (LangChain, `ChatPromptTemplate` + `with_structured_output`), guiada por
   uma matriz de **24 temas × 6 estilos de pergunta** (direta, comparação, suporte, caso de uso, objeção e
   informal), com personas sorteadas e a lista de perguntas já geradas no prompt para evitar repetições.
2. **Limpeza**: normalização, filtros de tamanho, remoção de duplicatas exatas e de quase-duplicatas
   (similaridade TF-IDF ≥ 0,85).
3. **Checagem automática de números** (valores em R$$ e percentuais que não estão no documento).
4. **LLM como juiz**: cada par comparado com o documento; os inconsistentes são descartados.
5. **Revisão humana** de uma amostra estratificada de 10% e correções pontuais.

$pipeline_table

## Uso

```python
from datasets import load_dataset

dataset = load_dataset("$dataset_repo_id", split="train")
```

## Limitações

Os dados são **sintéticos** e descrevem uma empresa fictícia. Não servem como fonte de informação financeira,
tributária ou regulatória real.
""")

PIPELINE_LABELS = {
    "pares_brutos": "Pares gerados (brutos)",
    "apos_limpeza_e_deduplicacao": "Após limpeza e deduplicação",
    "reprovados_pelo_juiz": "Reprovados pelo LLM juiz",
    "apos_juiz": "Após o LLM juiz",
    "amostra_revisao_manual": "Revisados manualmente",
    "correcoes_pontuais": "Correções pontuais",
    "dataset_final": "**Dataset final**",
}


def format_examples(questions: list[str], answers: list[str], limit: int = 4) -> str:
    """Format question/answer pairs as Markdown for the model card."""
    blocks = [
        f"**Pergunta:** {q}\n\n**Resposta:** {a}"
        for q, a in list(zip(questions, answers))[:limit]
    ]
    return "\n\n---\n\n".join(blocks)


def format_pipeline_table(summary: dict) -> str:
    """Format the dataset-construction funnel as a Markdown table."""
    rows = [
        f"| {label} | {summary[key]} |"
        for key, label in PIPELINE_LABELS.items()
        if key in summary
    ]
    if not rows:
        return ""
    return "| Etapa | Pares |\n|---|---|\n" + "\n".join(rows)


def build_model_card(
    config: Config,
    summary: dict,
    base_model: str,
    system_prompt: str,
    examples: str,
) -> str:
    """Render the model card (README.md) for the Hub model repository."""
    return MODEL_CARD_TEMPLATE.substitute(
        base_model=base_model,
        model_name=config.MODEL_NAME,
        model_repo_id=config.model_repo_id,
        dataset_repo_id=config.dataset_repo_id,
        n_train=summary["exemplos_treino"],
        n_eval=summary["exemplos_validacao"],
        lora_rank=config.LORA_RANK,
        lora_alpha=config.LORA_ALPHA,
        lora_dropout=config.LORA_DROPOUT,
        use_rslora=config.USE_RSLORA,
        target_modules=", ".join(f"`{m}`" for m in config.TARGET_MODULES),
        trainable_params=f"{summary['parametros_treinaveis']:,}",
        trainable_pct=summary["percentual_treinavel"],
        batch_size=summary["batch_size_efetivo"],
        per_device=config.BATCH_SIZE,
        grad_accum=config.GRAD_ACCUM,
        epochs=config.EPOCHS,
        steps=summary["passos_totais"],
        learning_rate=config.LEARNING_RATE,
        scheduler=config.LR_SCHEDULER,
        warmup=config.WARMUP_RATIO,
        max_seq_length=config.MAX_SEQ_LENGTH,
        gpu=summary["gpu"],
        runtime=summary["tempo_treino_min"],
        loss_start=summary["loss_treino_inicial"],
        loss_end=summary["loss_treino_final"],
        eval_loss=summary["melhor_eval_loss"],
        system_prompt=system_prompt,
        examples=examples,
    )


def build_dataset_card(
    config: Config,
    n_examples: int,
    example: dict,
    pipeline_summary: dict,
    has_metadata: bool,
) -> str:
    """Render the dataset card (README.md) for the Hub dataset repository."""
    metadata_note = (
        f"O arquivo `{config.METADATA_FILE}` traz as mesmas conversas com `id`, "
        "`topic` (tema) e `style` (estilo da pergunta), para rastreabilidade."
        if has_metadata
        else ""
    )
    return DATASET_CARD_TEMPLATE.substitute(
        n_examples=n_examples,
        data_file=config.DATA_FILE,
        model_repo_id=config.model_repo_id,
        dataset_repo_id=config.dataset_repo_id,
        example=json.dumps(example, ensure_ascii=False, indent=2),
        metadata_note=metadata_note,
        pipeline_table=format_pipeline_table(pipeline_summary),
    )


def push_to_hub(
    model: Any,
    tokenizer: Any,
    config: Config,
    model_card: str,
    dataset_card: str,
) -> None:
    """Push the LoRA adapters, tokenizer and dataset to the Hub.

    Args:
        model: The fine-tuned model whose LoRA adapters will be pushed.
        tokenizer: The tokenizer associated with the model.
        config: The training configuration (usernames, repos, paths).
        model_card: The README.md content for the model repository.
        dataset_card: The README.md content for the dataset repository.

    Returns:
        None. The artifacts are uploaded as a side effect.
    """
    token = get_hf_token()
    api = HfApi(token=token)

    print(f"Enviando modelo para: {config.model_repo_id} ...")
    model.push_to_hub(config.model_repo_id, token=token, private=config.HF_PRIVATE)
    tokenizer.push_to_hub(config.model_repo_id, token=token, private=config.HF_PRIVATE)
    api.upload_file(
        path_or_fileobj=model_card.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=config.model_repo_id,
        repo_type="model",
    )
    print(f"Modelo disponivel em: https://huggingface.co/{config.model_repo_id}")

    print(f"Enviando dataset para: {config.dataset_repo_id} ...")
    api.create_repo(
        config.dataset_repo_id,
        repo_type="dataset",
        private=config.HF_PRIVATE,
        exist_ok=True,
    )
    for filename in (config.DATA_FILE, config.METADATA_FILE):
        if config.path(filename).exists():
            api.upload_file(
                path_or_fileobj=str(config.path(filename)),
                path_in_repo=filename,
                repo_id=config.dataset_repo_id,
                repo_type="dataset",
            )
    api.upload_file(
        path_or_fileobj=dataset_card.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=config.dataset_repo_id,
        repo_type="dataset",
    )
    print(f"Dataset disponivel em: https://huggingface.co/datasets/{config.dataset_repo_id}")

A célula abaixo monta os dois cards e mostra uma prévia do model card antes de
publicar. Com `PUSH_TO_HUB = True`, ela envia os artefatos para o Hub.

In [ ]:
pipeline_summary_path = config.path(config.PIPELINE_SUMMARY_FILE)
pipeline_summary = (
    json.loads(pipeline_summary_path.read_text(encoding="utf-8"))
    if pipeline_summary_path.exists()
    else {}
)

model_card = build_model_card(
    config,
    training_summary,
    base_model=model.peft_config["default"].base_model_name_or_path,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
    examples=format_examples(TEST_QUESTIONS, answers_after),
)
dataset_card = build_dataset_card(
    config,
    n_examples=len(raw_dataset),
    example=raw_dataset[0],
    pipeline_summary=pipeline_summary,
    has_metadata=config.path(config.METADATA_FILE).exists(),
)
print(model_card[:3000])

if config.PUSH_TO_HUB:
    assert config.HF_USERNAME != "SEU_USUARIO_HF", "Preencha HF_USERNAME na classe Config."
    push_to_hub(model, tokenizer, config, model_card, dataset_card)

## Parte 2 - Inferência e chat interativo

Esta parte carrega o modelo **publicado no Hub** e pode ser executada de forma
independente da Parte 1, por exemplo em uma nova sessão do Colab: basta instalar o
Unsloth (seção 1.1) e executar as células abaixo. É também um teste de que o
repositório publicado funciona. Se estiver na mesma sessão da Parte 1 e faltar
memória na GPU, reinicie a sessão antes.

In [ ]:
import os
from typing import Any, Optional

from unsloth import FastLanguageModel
from transformers import TextStreamer


MODEL_ID = "EstudoAI/Indie-Qwen3-4B-LoRA"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
ENABLE_THINKING = False

DEFAULT_SYSTEM_PROMPT = (
    "Você é a assistente virtual da Indie, o banco de quem trabalha por conta própria. A Indie é uma fintech brasileira que oferece serviços financeiros para MEIs, microempresas de serviços, freelancers e profissionais liberais.\n"
    "\n"
    "Produtos da Indie:\n"
    "- Indie Conta PJ: conta digital para pessoa jurídica, com Pix, cartões e Indie Abre MEI\n"
    "- Indie Notas: emissão de nota fiscal de serviço (NFS-e)\n"
    "- Indie Global: recebimento de pagamentos do exterior em USD, EUR e GBP\n"
    "- Cofre Fiscal: reserva automática para impostos e DAS-MEI automático\n"
    "- Indie Contábil: contabilidade online com contador dedicado\n"
    "- Indie Futuro: Caixinhas, Salário Indie, Seguro Renda Protegida e Previdência\n"
    "- Indie Cobranças: links de pagamento, cobrança recorrente e Tap to Pay\n"
    "\n"
    "Planos: Essencial (grátis), Pro (R$ 39,90/mês) e Premium (R$ 99,90/mês).\n"
    "\n"
    "Responda em português, de forma clara, cordial e precisa. Não dê recomendações individuais de investimento nem planejamento tributário personalizado, nunca peça senhas ou códigos e, se não souber uma informação, oriente o cliente a falar com o atendimento."
)

A função `load_model` carrega o modelo base com os adaptadores LoRA publicados e já o
prepara para inferência.

In [ ]:
def load_model(model_id: str = MODEL_ID) -> tuple:
    """Load the base model with the fine-tuned LoRA adapters.

    Args:
        model_id: The Hugging Face Hub repository identifier of
            the fine-tuned model. Defaults to 'MODEL_ID'.

    Returns:
        A tuple '(model, tokenizer)' with the loaded model, ready
        for inference, and its associated tokenizer.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        token=os.environ.get("HF_TOKEN"),
    )
    FastLanguageModel.for_inference(model)
    model.generation_config.max_length = None
    return model, tokenizer

A função `chat` gera e transmite (streaming) a resposta do modelo para uma pergunta.

In [ ]:
def chat(
    model: Any,
    tokenizer: Any,
    question: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 512,
) -> None:
    """Generate and stream a single chat answer.

    Args:
        model: The fine-tuned language model, already prepared
            for inference.
        tokenizer: The tokenizer associated with the model.
        question: The user question to send to the model.
        system_prompt: An optional system prompt overriding the
            default persona. Defaults to None, in which case
            'DEFAULT_SYSTEM_PROMPT' is used.
        max_new_tokens: The maximum number of new tokens to
            generate. Defaults to 512.

    Returns:
        None. The generated answer is streamed to standard output
        as a side effect.
    """
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"Voce: {question}")
    print("Assistente: ", end="")
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        use_cache=True,
    )
    print("-" * 50)

A célula abaixo carrega o modelo do Hub e responde a algumas perguntas novas, que não
estão na base nem na comparação da Parte 1.

In [ ]:
hub_model, hub_tokenizer = load_model()

for question in [
    "Sou personal trainer e cobro mensalidade dos meus alunos. Como a Indie pode me ajudar?",
    "Quanto custa a declaração anual do MEI no plano Essencial?",
    "Se eu cancelar o plano anual no meio do ano, recebo algum dinheiro de volta?",
]:
    chat(hub_model, hub_tokenizer, question)

A função `main` executa um chat interativo no terminal, que termina com `sair`,
`exit`, `quit` ou `q`, como no notebook de aula.

In [ ]:
def main() -> None:
    """Run an interactive chat loop in the terminal.

    Repeatedly prompts the user for a question, streaming the
    model's answer until an exit command is entered.

    Returns:
        None.
    """
    print("Digite 'sair' para encerrar.\n")
    while True:
        question = input("Voce: ").strip()
        if question.lower() in {"sair", "exit", "quit", "q"}:
            print("Ate logo!")
            break
        if not question:
            continue
        chat(hub_model, hub_tokenizer, question)


if __name__ == "__main__":
    main()